### Setup: Install Libraries

This cell ensures that the `scikit-learn` library, a fundamental tool for machine learning, is installed and ready for use in our environment.

In [17]:
!pip install scikit-learn

### Data Loading and Preprocessing

This code block downloads and parses linguistic data from a CONLLU file. This format is standard for annotated text. The `parse_conllu` function extracts important details for each word, such as its ID, actual word form, Part-of-Speech (POS) tag, and its grammatical head (parent word) in the sentence.

In [18]:
import urllib.request

def parse_conllu(url):
    response = urllib.request.urlopen(url)
    lines = [line.decode('utf-8') for line in response.readlines()]

    sentences = []
    current_sentence = []

    for line in lines:
        line = line.strip()

        # An empty line marks the end of a sentence
        if not line:
            if current_sentence:
                sentences.append(current_sentence)
                current_sentence = []
            continue

        # Skip comment lines
        if line.startswith("#"):
            continue

        parts = line.split("\t")

        # Skip complex tokens (like contractions or ranges)
        if "-" in parts[0] or "." in parts[0]:
            continue

        word_info = {
            "id": int(parts[0]),
            "form": parts[1],
            "upos": parts[3],      # Part of Speech tag
            "head": int(parts[6]), # Parent word ID
            "deprel": parts[7]     # Dependency label
        }
        current_sentence.append(word_info)

    if current_sentence:
        sentences.append(current_sentence)

    return sentences

train_url = "https://raw.githubusercontent.com/UniversalDependencies/UD_English-EWT/refs/heads/master/en_ewt-ud-train.conllu"
sentences = parse_conllu(train_url)

print(f"Total sentences loaded: {len(sentences)}")
print("\nExample sentence (first sentence words):")
for word in sentences[0]:
    print(f"ID: {word['id']}, Word: {word['form']}, POS: {word['upos']}, Head: {word['head']}")

Total sentences loaded: 12544

Example sentence (first sentence words):
ID: 1, Word: Al, POS: PROPN, Head: 0
ID: 2, Word: -, POS: PUNCT, Head: 3
ID: 3, Word: Zaman, POS: PROPN, Head: 1
ID: 4, Word: :, POS: PUNCT, Head: 7
ID: 5, Word: American, POS: ADJ, Head: 6
ID: 6, Word: forces, POS: NOUN, Head: 7
ID: 7, Word: killed, POS: VERB, Head: 1
ID: 8, Word: Shaikh, POS: PROPN, Head: 7
ID: 9, Word: Abdullah, POS: PROPN, Head: 8
ID: 10, Word: al, POS: PROPN, Head: 8
ID: 11, Word: -, POS: PUNCT, Head: 12
ID: 12, Word: Ani, POS: PROPN, Head: 8
ID: 13, Word: ,, POS: PUNCT, Head: 15
ID: 14, Word: the, POS: DET, Head: 15
ID: 15, Word: preacher, POS: NOUN, Head: 8
ID: 16, Word: at, POS: ADP, Head: 18
ID: 17, Word: the, POS: DET, Head: 18
ID: 18, Word: mosque, POS: NOUN, Head: 15
ID: 19, Word: in, POS: ADP, Head: 21
ID: 20, Word: the, POS: DET, Head: 21
ID: 21, Word: town, POS: NOUN, Head: 18
ID: 22, Word: of, POS: ADP, Head: 23
ID: 23, Word: Qaim, POS: PROPN, Head: 21
ID: 24, Word: ,, POS: PUNCT, H

### Defining the Parser's Core Logic

Here, we set up the fundamental parts of our dependency parser. The `State` class keeps track of the parsing process (words processed, words waiting, and the dependencies found). The `get_oracle` function is like a perfect guide, telling the parser the single best next move to make at each step to build the correct dependency tree.

In [19]:
from collections import deque

class State:
    def __init__(self, sentence):
        # Stack starts with ROOT (id 0)
        self.stack = [0]
        # Buffer holds all word IDs [1, 2, 3...]
        self.buffer = deque([w["id"] for w in sentence])
        self.arcs = [] # Stores parsed (head, child, label) links

        # Quick lookup dictionary for POS tags
        self.id_to_word = {w["id"]: w for w in sentence}
        self.id_to_word[0] = {"upos": "ROOT"}

def get_oracle(state, gold_arcs):
    """Determines the correct next transition based on the gold standard tree."""
    if len(state.stack) >= 2:
        top = state.stack[-1]
        second = state.stack[-2]

        # Check for LEFT-ARC
        if (top, second) in gold_arcs:
            second_deps = [child for h, child, _ in gold_arcs if h == second]
            parsed_children = [child for h, c, _ in state.arcs if h == second]
            if all(dep in parsed_children for dep in second_deps):
                label = next(l for h, c, l in gold_arcs if h == top and c == second)
                return f"LEFT-ARC:{label}"

        # Check for RIGHT-ARC
        if (second, top) in gold_arcs:
            top_deps = [child for h, child, _ in gold_arcs if h == top]
            parsed_children = [child for h, c, _ in state.arcs if h == top]
            if all(dep in parsed_children for dep in top_deps):
                label = next(l for h, c, l in gold_arcs if h == second and c == top)
                return f"RIGHT-ARC:{label}"

    # Otherwise, SHIFT if buffer has words
    if len(state.buffer) > 0:
        return "SHIFT"

    return None

This cell defines two core components for a dependency parser:

*   **`State` class**: Represents the current state of the dependency parser, including the `stack` (words processed), `buffer` (words yet to be processed), and `arcs` (the dependency relations found so far). It also provides a quick lookup for word information based on their IDs.
*   **`get_oracle` function**: This function acts as an 'oracle' for the parser. Given a current parser `state` and the `gold_arcs` (the correct dependency relations for a sentence), it determines the single correct next transition (either `SHIFT`, `LEFT-ARC`, or `RIGHT-ARC`). It helps in training the parser by telling it the optimal move at each step.

### Generating Training Data

This code creates the training data for our dependency parser. It uses two main functions:

*   **`extract_features`**: Gathers important information (like Part-of-Speech tags) from the current state of the parser.
*   **`generate_training_data`**: Simulates parsing each sentence, uses an 'oracle' to find the correct actions, and collects these 'feature-action' pairs. This data will teach our model how to parse sentences.

In [20]:
def extract_features(state):
    """Extracts POS tag features from the current parser state."""
    stack = state.stack
    buffer = state.buffer
    lookup = state.id_to_word

    # 1. POS of top word on stack
    s1 = lookup[stack[-1]]["upos"] if len(stack) >= 1 else "NULL"
    # 2. POS of second word on stack
    s2 = lookup[stack[-2]]["upos"] if len(stack) >= 2 else "NULL"

    # 3. POS of first word in buffer
    b1 = lookup[buffer[0]]["upos"] if len(buffer) >= 1 else "NULL"
    # 4. POS of second word in buffer
    b2 = lookup[buffer[1]]["upos"] if len(buffer) >= 2 else "NULL"

    return {
        "stack_top": s1,
        "stack_second": s2,
        "buffer_first": b1,
        "buffer_second": b2
    }

def generate_training_data(sentences):
    """Runs the oracle over sentences to collect training samples."""
    X_features, y_actions = [], []

    for sentence in sentences:
        gold_arcs = {(w["head"], w["id"], w["deprel"]) for w in sentence}
        state = State(sentence)

        while state.buffer or len(state.stack) > 1:
            transition = get_oracle(state, gold_arcs)
            if not transition:
                break

            features = extract_features(state)
            X_features.append(features)
            y_actions.append(transition)

            if transition == "SHIFT":
                state.stack.append(state.buffer.popleft())
            elif transition.startswith("LEFT-ARC"):
                label = transition.split(":")[1]
                child = state.stack.pop(-2)
                head = state.stack[-1]
                state.arcs.append((head, child, label))
            elif transition.startswith("RIGHT-ARC"):
                label = transition.split(":")[1]
                child = state.stack.pop()
                head = state.stack[-1]
                state.arcs.append((head, child, label))

    return X_features, y_actions

X_train_raw, y_train = generate_training_data(sentences[:1])
print("Output generated successfully!")
print(f"Total training steps for first sentence: {len(y_train)}")
print("Sample Pair:", X_train_raw[0], "==>", y_train[0])

Output generated successfully!
Total training steps for first sentence: 29
Sample Pair: {'stack_top': 'ROOT', 'stack_second': 'NULL', 'buffer_first': 'PROPN', 'buffer_second': 'PUNCT'} ==> SHIFT
